# PaySense — Engine 1: Cash Flow Forecasting
**Goal:** Train ARIMA, Prophet, and LSTM on personal finance data. Select the model with the lowest RMSE on the held-out test set.

| Step | Description |
|------|-------------|
| 0 | Setup: install packages, mount Drive |
| 1 | Imports & configuration |
| 2 | Load & clean data (Kaggle CSV + Personal Excel) |
| 3 | Exploratory Data Analysis |
| 4 | Monthly aggregation & balance calculation |
| 5 | Train / test split (80/20, time-series aware) |
| 6 | ARIMA |
| 7 | Prophet |
| 8 | LSTM |
| 9 | Model comparison & winner selection |
| 10 | Demo: 3-month forecast on personal data (real RM) |
| 11 | Save winner model |

## 0. Setup

In [ ]:
!pip install -q pmdarima prophet openpyxl shap

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

# ─────────────────────────────────────────────────────────────
# UPDATE these paths if your Drive folder differs
DRIVE_ROOT    = '/content/drive/MyDrive/PaySense/'
KAGGLE_CSV    = DRIVE_ROOT + 'Personal_Finance_Dataset.csv'
PERSONAL_XLSX = DRIVE_ROOT + 'farah_spending_history.xlsx'
MODEL_DIR     = DRIVE_ROOT + 'models/'
# ─────────────────────────────────────────────────────────────

os.makedirs(MODEL_DIR, exist_ok=True)
print("Drive mounted. Model dir:", MODEL_DIR)

## 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings, joblib, json
from datetime import datetime
from dateutil.relativedelta import relativedelta
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

import random, tensorflow as tf
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("Libraries loaded. TF:", tf.__version__)

## 2. Load & Clean Data
- **Kaggle CSV** — synthetic 2020–2024, used for model training/testing
- **Personal Excel** — 12 monthly sheets May 2025–Apr 2026, real RM, used for demo

In [ ]:
# ── 2.1 Kaggle CSV ───────────────────────────────────────────
df_kaggle_raw = pd.read_csv(KAGGLE_CSV)
df_kaggle_raw.columns = df_kaggle_raw.columns.str.strip()
df_kaggle_raw['Date'] = pd.to_datetime(df_kaggle_raw['Date'])

# Fix Type column — Salary/Investment rows are wrongly labelled Expense in source data
INCOME_CATS = {'Salary', 'Investment', 'Other'}
df_kaggle_raw['Type'] = df_kaggle_raw['Category'].apply(
    lambda c: 'Income' if c in INCOME_CATS else 'Expense'
)

df_kaggle = df_kaggle_raw[['Date','Transaction Description','Category','Amount','Type']].copy()
df_kaggle.columns = ['date','description','category','amount','type']
df_kaggle['source'] = 'kaggle'

print(f"Kaggle: {len(df_kaggle):,} rows | {df_kaggle['date'].min().date()} to {df_kaggle['date'].max().date()}")
print(df_kaggle['type'].value_counts())
df_kaggle.head()

In [ ]:
# ── 2.2 Personal Excel (12 sheets, one per month) ───────────
EXCEL_EPOCH = pd.Timestamp('1899-12-30')
SHEET_ORDER = [
    'May','June','July','August','September','October',
    'November','December','Jan 26','Feb 26','Mar 26','Apr 26'
]

all_xl = pd.read_excel(PERSONAL_XLSX, sheet_name=None, header=None, engine='openpyxl')

dfs_personal = []
for sheet in SHEET_ORDER:
    if sheet not in all_xl:
        print(f"  '{sheet}' not found — skipping")
        continue
    df_s = all_xl[sheet].copy()
    # Row 0=title, 1=blank, 2=headers → data starts at row index 3
    df_s = df_s.iloc[3:].reset_index(drop=True)
    df_s = df_s.dropna(how='all')
    if df_s.shape[1] < 5:
        print(f"  '{sheet}' has <5 columns — skipping")
        continue
    df_s = df_s.iloc[:, :5].copy()
    df_s.columns = ['month_label','date_serial','description','amount','category']
    df_s['date_serial'] = pd.to_numeric(df_s['date_serial'], errors='coerce')
    df_s = df_s.dropna(subset=['date_serial'])
    df_s['date'] = EXCEL_EPOCH + pd.to_timedelta(df_s['date_serial'].astype(int), unit='D')
    df_s['amount'] = pd.to_numeric(df_s['amount'], errors='coerce').abs()
    df_s = df_s.dropna(subset=['amount'])
    df_s = df_s[df_s['amount'] > 0]
    df_s['type'] = 'Expense'
    df_s['source'] = 'personal'
    dfs_personal.append(df_s[['date','description','category','amount','type','source']])
    print(f"  '{sheet}': {len(df_s)} rows")

df_personal = pd.concat(dfs_personal, ignore_index=True)
df_personal['date'] = pd.to_datetime(df_personal['date'])
df_personal['category'] = df_personal['category'].astype(str).str.strip()

print(f"\nPersonal total: {len(df_personal):,} rows | "
      f"{df_personal['date'].min().date()} to {df_personal['date'].max().date()}")
df_personal.head(10)

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Kaggle monthly income vs expense
df_k_m = (df_kaggle.groupby([pd.Grouper(key='date', freq='MS'), 'type'])['amount']
          .sum().unstack(fill_value=0))
ax = axes[0, 0]
df_k_m.plot(kind='bar', ax=ax, color=['#e74c3c','#2ecc71'], alpha=0.8, width=0.9)
ax.set_title('Kaggle — Monthly Income vs Expense')
ax.tick_params(axis='x', labelsize=6, rotation=45)
ax.legend(['Expense','Income'], fontsize=8)

# Kaggle expense by category
ax = axes[0, 1]
kc = df_kaggle[df_kaggle['type']=='Expense'].groupby('category')['amount'].sum().sort_values()
kc.plot(kind='barh', ax=ax, color='#e74c3c', alpha=0.8)
ax.set_title('Kaggle — Expense by Category')

# Personal monthly expenses
df_p_m = df_personal.groupby(pd.Grouper(key='date', freq='MS'))['amount'].sum()
ax = axes[1, 0]
ax.bar(df_p_m.index, df_p_m.values, color='#3498db', alpha=0.8, width=25)
ax.set_title('Personal — Monthly Total Expenses (RM)')
ax.set_ylabel('RM')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

# Personal category breakdown
ax = axes[1, 1]
pc = df_personal.groupby('category')['amount'].sum().sort_values().tail(12)
pc.plot(kind='barh', ax=ax, color='#3498db', alpha=0.8)
ax.set_title('Personal — Top Spending Categories (RM)')

plt.suptitle('PaySense — Engine 1 EDA', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(MODEL_DIR + 'eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Monthly Aggregation & Balance Calculation

In [ ]:
# ── 4.1 Kaggle monthly net + cumulative balance ──────────────
kaggle_monthly = (df_kaggle
    .groupby([pd.Grouper(key='date', freq='MS'), 'type'])['amount']
    .sum().unstack(fill_value=0))
kaggle_monthly.columns.name = None
for col in ['Income','Expense']:
    if col not in kaggle_monthly.columns:
        kaggle_monthly[col] = 0
kaggle_monthly.columns = [c.lower() for c in kaggle_monthly.columns]
kaggle_monthly['net']     = kaggle_monthly['income'] - kaggle_monthly['expense']
kaggle_monthly['balance'] = kaggle_monthly['net'].cumsum()
print(f"Kaggle: {len(kaggle_monthly)} monthly rows")
kaggle_monthly.head()

In [ ]:
# ── 4.2 Personal monthly expenses + estimated income ─────────
# ─────────────────────────────────────────────────────────────
# UPDATE: set your actual average monthly income in RM
MONTHLY_INCOME_RM = 1500.0
# ─────────────────────────────────────────────────────────────

personal_monthly_exp = df_personal.groupby(pd.Grouper(key='date', freq='MS'))['amount'].sum()
personal_monthly = pd.DataFrame({'expense': personal_monthly_exp})
personal_monthly['income']  = MONTHLY_INCOME_RM
personal_monthly['net']     = personal_monthly['income'] - personal_monthly['expense']
personal_monthly['balance'] = personal_monthly['net'].cumsum()
print(f"Personal: {len(personal_monthly)} monthly rows")
print(personal_monthly.round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax1 = axes[0]
ax1.plot(kaggle_monthly.index, kaggle_monthly['balance'], color='#2c3e50', linewidth=2)
ax1.fill_between(kaggle_monthly.index, kaggle_monthly['balance'], alpha=0.1, color='#2c3e50')
ax1.axhline(0, color='red', linestyle='--', alpha=0.4)
ax1.set_title('Kaggle — Cumulative Balance (2020–2024)')
ax1.set_ylabel('Balance')
ax1.xaxis.set_major_locator(mdates.YearLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax2 = axes[1]
ax2.plot(personal_monthly.index, personal_monthly['balance'],
         color='#8e44ad', linewidth=2, marker='o', markersize=5)
ax2.fill_between(personal_monthly.index, personal_monthly['balance'], alpha=0.1, color='#8e44ad')
ax2.axhline(0, color='red', linestyle='--', alpha=0.4, label='Zero line')
ax2.set_title(f'Personal — Balance (RM, income=RM{MONTHLY_INCOME_RM:.0f}/mo)')
ax2.set_ylabel('Balance (RM)')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
ax2.xaxis.set_major_locator(mdates.MonthLocator())
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.savefig(MODEL_DIR + 'balance_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Train / Test Split
- Target: monthly net cashflow (income − expense)
- Split: 80% train, 20% test — time-ordered, no shuffling

In [ ]:
series = kaggle_monthly['net'].copy()
series.index = pd.DatetimeIndex(series.index, freq='MS')

n = len(series)
split_idx    = int(n * 0.8)
train_series = series.iloc[:split_idx]
test_series  = series.iloc[split_idx:]
HORIZON      = len(test_series)

assert train_series.index.max() < test_series.index.min(), "DATA LEAKAGE"
print(f"Total  : {n} months")
print(f"Train  : {len(train_series)} months  ({train_series.index[0].date()} → {train_series.index[-1].date()})")
print(f"Test   : {len(test_series)} months  ({test_series.index[0].date()} → {test_series.index[-1].date()})")
print("✓ No data leakage.")

plt.figure(figsize=(14, 4))
plt.plot(train_series.index, train_series.values, color='#27ae60', linewidth=2, label=f'Train ({len(train_series)} months)')
plt.plot(test_series.index,  test_series.values,  color='#e74c3c', linewidth=2, label=f'Test ({len(test_series)} months)')
plt.axvline(test_series.index[0], color='black', linestyle='--', alpha=0.5)
plt.axhline(0, color='grey', linestyle=':', alpha=0.4)
plt.title('Monthly Net Cashflow — Train / Test Split')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Shared evaluation function — same for all 3 models
def evaluate_forecast(actual, predicted, model_name):
    actual    = np.array(actual)
    predicted = np.array(predicted)
    mae  = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    nz   = actual != 0
    mape = float(np.mean(np.abs((actual[nz]-predicted[nz])/actual[nz]))*100) if nz.any() else float('nan')
    print(f"\n{'─'*42}  {model_name}")
    print(f"  MAE  : {mae:.4f}")
    print(f"  RMSE : {rmse:.4f}   ← primary selection metric")
    print(f"  MAPE : {mape:.2f}%")
    return {'model': model_name, 'MAE': round(mae,4), 'RMSE': round(rmse,4), 'MAPE': round(mape,2)}

## 6. Model 1 — ARIMA

In [ ]:
from statsmodels.tsa.stattools import adfuller
res = adfuller(train_series, autolag='AIC')
print(f"ADF Statistic : {res[0]:.4f}")
print(f"p-value       : {res[1]:.4f}")
for k,v in res[4].items(): print(f"Critical ({k}) : {v:.4f}")
if res[1] < 0.05:
    print("\n✓ STATIONARY — ARIMA can proceed without differencing.")
else:
    print("\n⚠ NON-STATIONARY — auto_arima will apply differencing (d≥1).")

In [ ]:
from pmdarima import auto_arima
print("Running auto_arima (1–3 min on Colab)…")
model_arima = auto_arima(
    train_series, seasonal=True, m=12, stepwise=True,
    information_criterion='aic', error_action='ignore',
    suppress_warnings=True, trace=True, random_state=SEED
)
print(f"\nOrder    : {model_arima.order}")
print(f"Seasonal : {model_arima.seasonal_order}")
print(model_arima.summary())

In [ ]:
arima_pred, arima_ci = model_arima.predict(n_periods=HORIZON, return_conf_int=True)
arima_forecast = pd.Series(arima_pred, index=test_series.index)
ci_lo = pd.Series(arima_ci[:,0], index=test_series.index)
ci_hi = pd.Series(arima_ci[:,1], index=test_series.index)

arima_metrics = evaluate_forecast(test_series, arima_forecast, 'ARIMA')

plt.figure(figsize=(14, 5))
plt.plot(train_series.index, train_series.values, color='#27ae60', linewidth=2, label='Train')
plt.plot(test_series.index,  test_series.values,  color='#2c3e50', linewidth=2.5, label='Actual')
plt.plot(arima_forecast.index, arima_forecast.values, color='#e74c3c', linewidth=2,
         linestyle='--', label=f'ARIMA (RMSE={arima_metrics["RMSE"]:.2f})')
plt.fill_between(test_series.index, ci_lo, ci_hi, alpha=0.15, color='#e74c3c', label='95% CI')
plt.axhline(0, color='grey', linestyle=':', alpha=0.4)
plt.title('ARIMA — Forecast vs Actual'); plt.ylabel('Net Cashflow'); plt.legend()
plt.tight_layout()
plt.savefig(MODEL_DIR+'arima_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Model 2 — Prophet

In [ ]:
from prophet import Prophet

df_prophet_train = pd.DataFrame({'ds': train_series.index, 'y': train_series.values})
model_prophet = Prophet(
    yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False,
    seasonality_mode='additive', changepoint_prior_scale=0.05
)
model_prophet.fit(df_prophet_train)
print("Prophet fitted.")

In [ ]:
future        = model_prophet.make_future_dataframe(periods=HORIZON, freq='MS')
prophet_full  = model_prophet.predict(future)
prophet_forecast = (prophet_full.set_index('ds')['yhat'].iloc[-HORIZON:])
prophet_forecast.index = test_series.index

prophet_metrics = evaluate_forecast(test_series, prophet_forecast, 'Prophet')

plt.figure(figsize=(14, 5))
plt.plot(train_series.index, train_series.values, color='#27ae60', linewidth=2, label='Train')
plt.plot(test_series.index,  test_series.values,  color='#2c3e50', linewidth=2.5, label='Actual')
plt.plot(prophet_forecast.index, prophet_forecast.values, color='#f39c12', linewidth=2,
         linestyle='--', label=f'Prophet (RMSE={prophet_metrics["RMSE"]:.2f})')
yhat_lo = prophet_full.set_index('ds')['yhat_lower'].iloc[-HORIZON:].values
yhat_hi = prophet_full.set_index('ds')['yhat_upper'].iloc[-HORIZON:].values
plt.fill_between(test_series.index, yhat_lo, yhat_hi, alpha=0.15, color='#f39c12', label='80% CI')
plt.axhline(0, color='grey', linestyle=':', alpha=0.4)
plt.title('Prophet — Forecast vs Actual'); plt.ylabel('Net Cashflow'); plt.legend()
plt.tight_layout()
plt.savefig(MODEL_DIR+'prophet_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

model_prophet.plot_components(prophet_full)
plt.tight_layout(); plt.show()

## 8. Model 3 — LSTM

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

WINDOW_SIZE = 6   # look back 6 months to predict the next

scaler_lstm  = MinMaxScaler(feature_range=(0,1))
train_scaled = scaler_lstm.fit_transform(train_series.values.reshape(-1,1))
test_scaled  = scaler_lstm.transform(test_series.values.reshape(-1,1))

def make_sequences(data, w):
    X, y = [], []
    for i in range(len(data)-w):
        X.append(data[i:i+w, 0])
        y.append(data[i+w, 0])
    return np.array(X), np.array(y)

X_train, y_train = make_sequences(train_scaled, WINDOW_SIZE)
full_scaled      = np.concatenate([train_scaled, test_scaled])
X_test,  y_test  = make_sequences(full_scaled[-(HORIZON+WINDOW_SIZE):], WINDOW_SIZE)

X_train = X_train.reshape(-1, WINDOW_SIZE, 1)
X_test  = X_test.reshape(-1,  WINDOW_SIZE, 1)
print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")

In [ ]:
model_lstm = Sequential([
    LSTM(64, input_shape=(WINDOW_SIZE,1), return_sequences=True),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
], name='Engine1_LSTM')

model_lstm.compile(optimizer='adam', loss='mse', metrics=['mae'])
model_lstm.summary()

callbacks = [
    EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6, verbose=1)
]

history = model_lstm.fit(
    X_train, y_train, epochs=200, batch_size=8,
    validation_split=0.15, callbacks=callbacks, verbose=1
)

plt.figure(figsize=(10,4))
plt.plot(history.history['loss'],     color='#2c3e50', label='Train Loss')
plt.plot(history.history['val_loss'], color='#e74c3c', label='Val Loss')
plt.title('LSTM — Training Loss'); plt.xlabel('Epoch'); plt.ylabel('MSE'); plt.legend()
plt.tight_layout()
plt.savefig(MODEL_DIR+'lstm_training_loss.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Stopped at epoch {len(history.history['loss'])}")

In [ ]:
lstm_pred_scaled = model_lstm.predict(X_test, verbose=0)
lstm_forecast    = pd.Series(
    scaler_lstm.inverse_transform(lstm_pred_scaled).flatten()[:HORIZON],
    index=test_series.index
)
lstm_metrics = evaluate_forecast(test_series, lstm_forecast, 'LSTM')

plt.figure(figsize=(14,5))
plt.plot(train_series.index, train_series.values, color='#27ae60', linewidth=2, label='Train')
plt.plot(test_series.index,  test_series.values,  color='#2c3e50', linewidth=2.5, label='Actual')
plt.plot(lstm_forecast.index, lstm_forecast.values, color='#9b59b6', linewidth=2,
         linestyle='--', label=f'LSTM (RMSE={lstm_metrics["RMSE"]:.2f})')
plt.axhline(0, color='grey', linestyle=':', alpha=0.4)
plt.title('LSTM — Forecast vs Actual'); plt.ylabel('Net Cashflow'); plt.legend()
plt.tight_layout()
plt.savefig(MODEL_DIR+'lstm_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Model Comparison & Winner Selection

In [ ]:
results_df = pd.DataFrame([arima_metrics, prophet_metrics, lstm_metrics])
results_df = results_df.sort_values('RMSE').reset_index(drop=True)
results_df.insert(0, 'Rank', range(1, len(results_df)+1))
results_df['Winner'] = ['✓ SELECTED' if i==0 else '' for i in range(len(results_df))]

print("\n" + "="*52)
print("   ENGINE 1 — MODEL EVALUATION RESULTS")
print("="*52)
print(results_df.to_string(index=False))
print("="*52)
WINNER_NAME = results_df.iloc[0]['model']
print(f"\nWinner: {WINNER_NAME}  (RMSE={results_df.iloc[0]['RMSE']})")

In [ ]:
plt.figure(figsize=(14,6))
plt.plot(train_series.index, train_series.values, color='#95a5a6', linewidth=1.5, alpha=0.6, label='Train')
plt.plot(test_series.index,  test_series.values,  color='#2c3e50', linewidth=2.5, zorder=5, label='Actual')
plt.plot(arima_forecast.index,    arima_forecast.values,    color='#e74c3c', linewidth=2,
         linestyle='--', label=f'ARIMA   RMSE={arima_metrics["RMSE"]:.2f}')
plt.plot(prophet_forecast.index,  prophet_forecast.values,  color='#f39c12', linewidth=2,
         linestyle='--', label=f'Prophet RMSE={prophet_metrics["RMSE"]:.2f}')
plt.plot(lstm_forecast.index,     lstm_forecast.values,     color='#9b59b6', linewidth=2,
         linestyle='--', label=f'LSTM    RMSE={lstm_metrics["RMSE"]:.2f}')
plt.axvline(test_series.index[0], color='black', linestyle=':', alpha=0.3)
plt.title(f'Engine 1 — All Forecasts vs Actual  |  Winner: {WINNER_NAME}', fontweight='bold')
plt.ylabel('Monthly Net Cashflow'); plt.legend()
plt.tight_layout()
plt.savefig(MODEL_DIR+'engine1_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
results_df.to_csv(MODEL_DIR+'engine1_results.csv', index=False)
print("Comparison chart and results table saved.")

## 10. Demo — 3-Month Forecast on Personal Data (Real RM)

In [ ]:
personal_net = personal_monthly['net'].dropna()
personal_net.index = pd.DatetimeIndex(personal_net.index, freq='MS')
FORECAST_MONTHS = 3
forecast_dates  = pd.date_range(
    start=personal_net.index[-1] + relativedelta(months=1),
    periods=FORECAST_MONTHS, freq='MS'
)
print(f"Personal history: {len(personal_net)} months "
      f"({personal_net.index[0].strftime('%b %Y')} → {personal_net.index[-1].strftime('%b %Y')})")
print(f"Forecasting: {forecast_dates[0].strftime('%b %Y')} → {forecast_dates[-1].strftime('%b %Y')}")

In [ ]:
# Refit winner on personal data, then forecast 3 months ahead
if WINNER_NAME == 'ARIMA':
    from pmdarima import auto_arima as _aa
    _m = _aa(personal_net, seasonal=True, m=12, stepwise=True,
             suppress_warnings=True, error_action='ignore', random_state=SEED)
    demo_vals = _m.predict(n_periods=FORECAST_MONTHS)

elif WINNER_NAME == 'Prophet':
    from prophet import Prophet as _P
    _df = pd.DataFrame({'ds': personal_net.index, 'y': personal_net.values})
    _m  = _P(yearly_seasonality=False, weekly_seasonality=False, daily_seasonality=False)
    _m.fit(_df)
    _fut = _m.make_future_dataframe(periods=FORECAST_MONTHS, freq='MS')
    demo_vals = _m.predict(_fut)['yhat'].values[-FORECAST_MONTHS:]

elif WINNER_NAME == 'LSTM':
    from sklearn.preprocessing import MinMaxScaler as _S
    _sc  = _S()
    _ps  = _sc.fit_transform(personal_net.values.reshape(-1,1))
    _seed = _ps[-WINDOW_SIZE:].reshape(1, WINDOW_SIZE, 1)
    demo_vals = []
    for _ in range(FORECAST_MONTHS):
        _p = model_lstm.predict(_seed, verbose=0)[0, 0]
        demo_vals.append(_p)
        _seed = np.roll(_seed, -1, axis=1)
        _seed[0, -1, 0] = _p
    demo_vals = _sc.inverse_transform(np.array(demo_vals).reshape(-1,1)).flatten()

demo_forecast = pd.Series(demo_vals, index=forecast_dates)
proj_balance  = personal_monthly['balance'].iloc[-1] + demo_forecast.cumsum()

print(f"\n3-Month Projection  ({WINNER_NAME})")
print(f"{'─'*55}")
print(f"{'Month':<12} {'Net (RM)':>12} {'Proj. Balance':>16} {'Status':>8}")
print(f"{'─'*55}")
for dt, net, bal in zip(forecast_dates, demo_forecast, proj_balance):
    status = '⚠ RISK' if bal < 100 else '✓ OK'
    print(f"{dt.strftime('%b %Y'):<12} {net:>+12.2f} {bal:>16.2f} {status:>8}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 9))

ax1.bar(personal_net.index, personal_net.values, width=25, color='#3498db', alpha=0.75, label='Historical Net')
ax1.bar(forecast_dates, demo_forecast.values,    width=25, color='#f39c12', alpha=0.9,
        label=f'{WINNER_NAME} Forecast')
ax1.axhline(0, color='red', linestyle='--', alpha=0.4)
ax1.set_title('Personal Net Cashflow + 3-Month Forecast', fontweight='bold')
ax1.set_ylabel('Net (RM)'); ax1.legend()
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
ax1.xaxis.set_major_locator(mdates.MonthLocator())
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45)

ax2.plot(personal_monthly['balance'].index, personal_monthly['balance'].values,
         color='#8e44ad', linewidth=2, marker='o', markersize=5, label='Historical Balance')
ax2.plot(proj_balance.index, proj_balance.values, color='#f39c12', linewidth=2.5,
         linestyle='--', marker='s', markersize=7, label='Projected Balance')
ax2.fill_between(proj_balance.index, 0, proj_balance.values,
                 where=(proj_balance.values < 0), alpha=0.3, color='red', label='Deficit zone')
ax2.axhline(0,   color='red',    linestyle='--', alpha=0.4)
ax2.axhline(100, color='orange', linestyle=':',  alpha=0.4, label='RM100 safety buffer')
ax2.set_title('Projected Bank Balance — Next 3 Months', fontweight='bold')
ax2.set_ylabel('Balance (RM)'); ax2.legend()
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
ax2.xaxis.set_major_locator(mdates.MonthLocator())
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.savefig(MODEL_DIR+'engine1_personal_demo.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Save Winner Model

In [ ]:
if WINNER_NAME == 'ARIMA':
    joblib.dump(model_arima, MODEL_DIR+'engine1_winner.pkl')
    print("Saved: engine1_winner.pkl  (ARIMA)")
elif WINNER_NAME == 'Prophet':
    joblib.dump(model_prophet, MODEL_DIR+'engine1_winner.pkl')
    print("Saved: engine1_winner.pkl  (Prophet)")
elif WINNER_NAME == 'LSTM':
    model_lstm.save(MODEL_DIR+'engine1_winner.h5')
    joblib.dump(scaler_lstm, MODEL_DIR+'engine1_scaler.pkl')
    print("Saved: engine1_winner.h5  (LSTM)")
    print("Saved: engine1_scaler.pkl (MinMaxScaler)")

meta = {
    'engine': 1, 'task': 'cash_flow_forecasting', 'winner': WINNER_NAME,
    'selection_criterion': 'lowest_RMSE',
    'results': {'ARIMA': arima_metrics, 'Prophet': prophet_metrics, 'LSTM': lstm_metrics},
    'data': {
        'primary':   'Personal_Finance_Dataset.csv (Kaggle 2020-2024)',
        'personal':  'farah_spending_history.xlsx (May 2025 - Mar 2026)',
        'train':     f"{train_series.index[0].date()} to {train_series.index[-1].date()}",
        'test':      f"{test_series.index[0].date()} to {test_series.index[-1].date()}",
    },
    'generated_at': datetime.now().isoformat()
}
with open(MODEL_DIR+'engine1_metadata.json','w') as f:
    json.dump(meta, f, indent=2)

results_df.to_csv(MODEL_DIR+'engine1_results.csv', index=False)

print("\n✓ All Engine 1 outputs saved to:", MODEL_DIR)
print("\n" + "="*50)
print(f"  ENGINE 1 COMPLETE — Winner: {WINNER_NAME}")
print(f"  RMSE: {results_df.iloc[0]['RMSE']}")
print(f"  Next: run paysense_engine2_classification.ipynb")
print("="*50)